# 語料處理

https://huggingface.co/datasets/BelleGroup/train_3.5M_CN

        指令是否簡短一些較好?還是沒差異??

        #此處: 開頭do not add bos_token_id  Why???為何不加上bos_token??
        
        tokenizer.padding_side is left
        最後面有一個eos_token符號</s>  好處:產生文字時知道何時該結束!
        注意:文章太長被截斷時，最後一個編碼沒有eos_token (會產生問題嗎?)


        Tokenize
        每個樣本長度不一，因此編碼後的長度不一，若有超出最大長度者會被截斷。文字長度訂定長一些，盡量不要被截斷。
        
        此處編碼不進行padding,padding工作交給後續的data_collator，進行padding即可(靠左)，每批次的長度以該批長度最長者為該批之整體句子長度。好處是:訓練時有些批之句子長度較短，較有效率。
        
        labels: (1)在labels處理時，Assistant的回答保留其ids，會將其他非output之字串以-100取代，(2)會在collator做批量時同時做padding(靠左)，labels之<pad>位置會被塞入"-100"

        attention_mask的值皆為1，若為<pad>則是0

        https://github.com/A-baoYang/alpaca-7b-chinese


資料轉成對話格式:     

        Human: 
        你好

        Assistant: 

        你好，有什么可以帮助你的吗？
        Human: 
        今天天气怎么样？

        Assistant: 

        不好意思，我无法回答你的问题，因为我不知道你的位置信息，同时我目前还无法获取到最新的天气信息。
        55
        Human: 
        如何學習程式?

        Assistant: 

        你可以修讀資管系的程式設計課程

In [2]:
from itertools import chain
from typing import Any, Callable, Dict, List
import copy
from transformers import PreTrainedTokenizer
import json

IGNORE_INDEX = -100

In [3]:
import torch
import transformers
from transformers import BloomTokenizerFast, BloomForCausalLM, TrainingArguments
import datasets

In [4]:
from transformers import BloomTokenizerFast, BloomForCausalLM, TrainingArguments,AutoTokenizer
import datasets

In [5]:
tokenizer = AutoTokenizer.from_pretrained('Langboat/bloom-389m-zh')
# tokenizer = AutoTokenizer.from_pretrained('YeungNLP/bloomz-396m-zh', use_fast=True)

In [6]:
tokenizer.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '</s>',
 'unk_token': '<unk>',
 'pad_token': '<pad>'}

In [7]:
tokenizer.padding_side

'left'

In [8]:

def sft_sample_to_ids(conversations: Dict[str, Any], tokenizer: PreTrainedTokenizer):
    input_ids = []
    labels = []
    for sentence in conversations:
        # print(sentence)
        sentence_from = sentence["from"].lower()
        sentence_value = (
            "Human: \n" + sentence["value"] + "\n\nAssistant: \n"
            if sentence_from == "human"
            else sentence["value"]
        )  # https://github.com/LianjiaTech/BELLE/issues/337
        # print(sentence_value)
        # conversation += sentence_value
        sentence_ids = tokenizer.encode(
            sentence_value, add_special_tokens=False
        )  # do not add bos_token_id
        label = (
            copy.deepcopy(sentence_ids)
            if sentence_from != "human"
            else [IGNORE_INDEX] * len(sentence_ids)
        )
        input_ids += sentence_ids
        labels += label
        # add eos at every end of assistant sentence
        if sentence_from != "human":
            input_ids += [tokenizer.eos_token_id]  # make sure eos_token_id is correct
            labels += [tokenizer.eos_token_id]
    return input_ids, labels


### 這些數字代表甚麼?
    23069 Human
    29 :
    2705 \n
    2 </s>
    4122, 15263  Assistant

    4122, 15263, 29, 2705  Assistant:\n


In [20]:
data_point =  {
        "id": "uniq_sample_id",
        "conversations": [
            {"from": "human", "value": "你好"},
            {"from": "assistant", "value": "你好，有什么可以帮助你的吗？"},
            {"from": "human", "value": "今天天气怎么样？"},
            {"from": "assistant", "value": "不好意思，我无法回答你的问题，因为我不知道你的位置信息，同时我目前还无法获取到最新的天气信息。"}
        ]
    }

In [21]:
sft_sample_to_ids(data_point["conversations"],tokenizer)

([23069,
  29,
  2705,
  11877,
  189,
  189,
  4122,
  15263,
  29,
  2705,
  11877,
  355,
  11137,
  34653,
  3515,
  4582,
  2,
  23069,
  29,
  2705,
  6089,
  17216,
  28609,
  189,
  189,
  4122,
  15263,
  29,
  2705,
  25199,
  355,
  683,
  5684,
  10639,
  3515,
  1960,
  355,
  3047,
  14202,
  3515,
  6857,
  3882,
  355,
  4232,
  683,
  4415,
  1624,
  5684,
  12801,
  851,
  24399,
  17216,
  3882,
  420,
  2],
 [-100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  11877,
  355,
  11137,
  34653,
  3515,
  4582,
  2,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  25199,
  355,
  683,
  5684,
  10639,
  3515,
  1960,
  355,
  3047,
  14202,
  3515,
  6857,
  3882,
  355,
  4232,
  683,
  4415,
  1624,
  5684,
  12801,
  851,
  24399,
  17216,
  3882,
  420,
  2])

In [13]:
tokenizer.decode([23069, 29])

'Human:'

In [19]:
tokenizer.decode([678, 15263,29])

'assistant:'

In [14]:
tokenizer.decode([2])

'</s>'

In [24]:
# 4122, 15263,    29,  2705 Assistant:\n
tokenizer(['Assistant'])

{'input_ids': [[4122, 15263]], 'attention_mask': [[1, 1]]}

In [23]:
tokenizer(['Human'])

{'input_ids': [[23069]], 'attention_mask': [[1]]}

In [22]:
tokenizer.decode([23069,
  29,
  2705,
  11877,
  189,
  189,
  4122,
  15263,
  29,
  2705,
  11877,
  355,
  11137,
  34653,
  3515,
  4582,
  2])

'Human: \n你好\n\nAssistant: \n你好，有什么可以帮助你的吗？</s>'

In [25]:
tokenizer.decode([23069,
  29,
  2705,
  6089,
  17216,
  28609,
  189,
  189,
  4122,
  15263,
  29,
  2705,
  25199,
  355,
  683,
  5684,
  10639,
  3515,
  1960,
  355,
  3047,
  14202,
  3515,
  6857,
  3882,
  355,
  4232,
  683,
  4415,
  1624,
  5684,
  12801,
  851,
  24399,
  17216,
  3882,
  420,
  2])

'Human: \n今天天气怎么样？\n\nAssistant: \n不好意思，我无法回答你的问题，因为我不知道你的位置信息，同时我目前还无法获取到最新的天气信息。</s>'

In [ ]:
# label答案 Assistant的回答

tokenizer.decode([11877,
  355,
  11137,
  34653,
  3515,
  4582,
  2])

'你好，有什么可以帮助你的吗？</s>'

In [27]:
# label答案 Assistant的回答

tokenizer.decode([25199,
  355,
  683,
  5684,
  10639,
  3515,
  1960,
  355,
  3047,
  14202,
  3515,
  6857,
  3882,
  355,
  4232,
  683,
  4415,
  1624,
  5684,
  12801,
  851,
  24399,
  17216,
  3882,
  420,
  2])

'不好意思，我无法回答你的问题，因为我不知道你的位置信息，同时我目前还无法获取到最新的天气信息。</s>'

In [10]:
a=False
"A" if a==True else "B"

'B'

In [11]:
dataset_samples = [
    {
        "id": "66623028",
        "conversations": [
            {
                "from": "human",
                "value": "归纳介绍三种不同类型的电影题材"
            },
            {
                "from": "assistant",
                "value": "1. 动作片：动作片通常强调动作和战斗场面，故事情节紧凑，充满了紧张和兴奋的氛围。主角一般是具有超凡实力或特殊技能的英雄人物，往往会与反派角色进行一系列激烈的格斗，追逐和爆破等酷炫场面。例如《速度与激情》和《复仇者联盟》等。\n2. 爱情片：爱情片是以爱情为主题的电影，着重呈现角色之间的感情纠葛，展现了爱情的美好和悲壮。情节与角色之间的发展令人深思，增加了观众的感性体验。这种电影的故事情节主要以人类与人类之间的情感变化为主题。例如《泰坦尼克号》和《让子弹飞》等。\n3. 科幻片：科幻片是以未来科技和未知世界为主题的电影，通常包括时间旅行、外星人、虚拟现实、未来社会等各种奇幻的元素。这种电影描绘了一种比现实更加宏伟、更加神秘和惊奇的世界。例如《星际穿越》和《发条橙》等。"
            }
        ]
    },
    {
        "id": "18205437",
        "conversations": [
            {
                "from": "human",
                "value": "根据给定的食材，编写一道美食菜谱。\n鸡腿肉、蘑菇、洋葱"
            },
            {
                "from": "assistant",
                "value": "鸡肉炒蘑菇洋葱\n食材：\n- 鸡腿肉 400克，去骨去皮切成小块\n- 蘑菇 200克，切片\n- 洋葱 1个，切丝\n- 生姜 适量，切丝\n- 酱油 2勺\n- 料酒 1勺\n- 盐 适量\n- 白胡椒粉 适量\n- 植物油 适量\n做法：\n1. 在一个碗里放入切好的鸡腿肉块，加入酱油、料酒和适量的白胡椒粉，搅拌均匀，腌制10分钟。\n2. 热锅凉油，加入姜丝翻炒，放入腌好的鸡肉块翻炒至变色。\n3. 加入洋葱丝和蘑菇，翻炒2-3分钟至蘑菇变软。\n4. 加入适量的盐和少量的水，盖上锅盖焖煮5分钟直至鸡肉熟透。\n5. 最后翻炒几下，使汤汁变浓稠即可。\n温馨提示：\n1. 在腌制鸡肉时，如果想让肉质更嫩滑，可以加入少量的淀粉和蛋清。\n2. 可以根据个人口味加入其他调料，如蒜末、辣椒等。\n3. 洋葱和蘑菇可以用其他蔬菜代替，如青椒、胡萝卜等。"
            },
            {
                "from": "human",
                "value": "太好了，我会试着照着你教的做一次，但是我最近买了一部新相机，想让你推荐一些好的拍照地点。"
            },
            {
                "from": "assistant",
                "value": "当地的风景区、公园、景点等等都是比较适合拍照的地点。如果你想要一些特别的地方，可以考虑一些建筑群，街道巷子、咖啡馆、书店等具有文艺气息的地方。"
            },
            {
                "from": "human",
                "value": "嗯，这些地方都很不错，但是我还想找一些可以体验当地文化和风俗的地点，你有什么好的建议吗？"
            },
            {
                "from": "assistant",
                "value": "如果你想要更深入地了解当地文化和风俗，可以考虑参加一些游学活动，比如参观当地的博物馆、参加民俗节庆、尝试当地美食等等。此外，也可以去一些小城镇，感受那里的古朴风情。"
            }
        ]
    }
]


In [12]:
sft_sample_to_ids(dataset_samples[0]["conversations"],tokenizer)

([23069,
  29,
  2705,
  38857,
  6910,
  21954,
  3496,
  20885,
  6213,
  25762,
  189,
  189,
  4122,
  15263,
  29,
  2705,
  20,
  17,
  210,
  13566,
  2740,
  1022,
  13566,
  2740,
  6250,
  6707,
  13566,
  641,
  9800,
  32843,
  355,
  6398,
  23397,
  5355,
  26000,
  355,
  33036,
  13464,
  641,
  20027,
  373,
  29169,
  420,
  11669,
  25057,
  4830,
  3316,
  7015,
  17607,
  1298,
  8116,
  10322,
  373,
  9784,
  6929,
  355,
  9629,
  739,
  1210,
  2133,
  2777,
  7759,
  2463,
  14897,
  31269,
  1575,
  5381,
  355,
  33077,
  641,
  5433,
  3825,
  1271,
  8771,
  24496,
  32843,
  420,
  5160,
  1125,
  7882,
  1210,
  27967,
  14636,
  30841,
  1248,
  6322,
  12221,
  671,
  21,
  17,
  210,
  14090,
  2740,
  1022,
  14090,
  2740,
  11200,
  14090,
  32990,
  31938,
  355,
  17054,
  17645,
  7759,
  7279,
  13737,
  13586,
  9840,
  355,
  20919,
  657,
  14090,
  17094,
  1288,
  641,
  9187,
  12342,
  420,
  23397,
  1210,
  7759,
  7279,
  2220,
  8507

In [13]:
sft_sample_to_ids(dataset_samples[1]["conversations"],tokenizer)

([23069,
  29,
  2705,
  3442,
  2391,
  5178,
  32746,
  355,
  10689,
  16265,
  20628,
  5794,
  12665,
  671,
  8700,
  9819,
  5179,
  553,
  33057,
  553,
  37507,
  189,
  189,
  4122,
  15263,
  29,
  2705,
  36934,
  12406,
  33057,
  37507,
  189,
  32746,
  1022,
  189,
  16,
  210,
  8700,
  9819,
  5179,
  5858,
  1517,
  355,
  1400,
  5817,
  1400,
  3870,
  26242,
  1200,
  6800,
  189,
  16,
  210,
  33057,
  825,
  1517,
  355,
  34595,
  189,
  16,
  210,
  37507,
  404,
  895,
  355,
  2748,
  7142,
  189,
  16,
  23060,
  11055,
  210,
  19566,
  355,
  2748,
  7142,
  189,
  16,
  210,
  30829,
  415,
  24674,
  189,
  16,
  210,
  38053,
  404,
  24674,
  189,
  16,
  210,
  9295,
  210,
  19566,
  189,
  16,
  16231,
  29316,
  6128,
  210,
  19566,
  189,
  16,
  210,
  8503,
  3793,
  210,
  19566,
  189,
  7398,
  1022,
  189,
  20,
  17,
  3454,
  1617,
  13962,
  1234,
  14489,
  2748,
  4509,
  8700,
  9819,
  5179,
  6800,
  355,
  5067,
  30829,
  553,
 

In [14]:
tokenizer("Human:")

{'input_ids': [23069, 29], 'attention_mask': [1, 1]}

In [15]:
model_max_length=512

def generate_and_tokenize_prompt(
    model_max_length: int,
    tokenizer: PreTrainedTokenizer,
    data_point: Dict[str, Any],
    fix_length=False,
    padding_side="left",
):
    conversations = data_point["conversations"]
    input_ids, labels = sft_sample_to_ids(conversations, tokenizer)

    input_ids = input_ids[:model_max_length]
    labels = labels[:model_max_length]

    if all(x == IGNORE_INDEX for x in labels):
        labels[18:24] = input_ids[
            18:24
        ]  # labels can not have all values being -100. 18 and 24 are just random numbers
    attention_mask = [1] * len(input_ids)

    if fix_length and model_max_length > len(input_ids):
        if padding_side == "left":
            input_ids = [tokenizer.pad_token_id] * (
                model_max_length - len(input_ids)
            ) + input_ids
            labels = [tokenizer.pad_token_id] * (
                model_max_length - len(labels)
            ) + labels
            attention_mask = [0] * (
                model_max_length - len(attention_mask)
            ) + attention_mask
        else:
            input_ids = input_ids + [tokenizer.pad_token_id] * (
                model_max_length - len(input_ids)
            )
            labels = labels + [tokenizer.pad_token_id] * (
                model_max_length - len(labels)
            )
            attention_mask = attention_mask + [0] * (
                model_max_length - len(attention_mask)
            )

    tokenized_full_prompt = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }
    return tokenized_full_prompt

In [16]:
data_point =  {
        "id": "uniq_sample_id",
        "conversations": [
            {"from": "human", "value": "你好"},
            {"from": "assistant", "value": "你好，有什么可以帮助你的吗？"},
            {"from": "human", "value": "今天天气怎么样？"},
            {"from": "assistant", "value": "不好意思，我无法回答你的问题，因为我不知道你的位置信息，同时我目前还无法获取到最新的天气信息。"}
        ]
    }

In [17]:
sft_sample_to_ids(conversations = data_point["conversations"],tokenizer=tokenizer)

([23069,
  29,
  2705,
  11877,
  189,
  189,
  4122,
  15263,
  29,
  2705,
  11877,
  355,
  11137,
  34653,
  3515,
  4582,
  2,
  23069,
  29,
  2705,
  6089,
  17216,
  28609,
  189,
  189,
  4122,
  15263,
  29,
  2705,
  25199,
  355,
  683,
  5684,
  10639,
  3515,
  1960,
  355,
  3047,
  14202,
  3515,
  6857,
  3882,
  355,
  4232,
  683,
  4415,
  1624,
  5684,
  12801,
  851,
  24399,
  17216,
  3882,
  420,
  2],
 [-100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  11877,
  355,
  11137,
  34653,
  3515,
  4582,
  2,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  25199,
  355,
  683,
  5684,
  10639,
  3515,
  1960,
  355,
  3047,
  14202,
  3515,
  6857,
  3882,
  355,
  4232,
  683,
  4415,
  1624,
  5684,
  12801,
  851,
  24399,
  17216,
  3882,
  420,
  2])

In [18]:
tokenizer.decode([13253,
  355,
  12398,
  39180,
  3678,
  4854,
  2,])

' END，MD 这次ulu岁</s>'

In [19]:
generate_and_tokenize_prompt(data_point=data_point, tokenizer=tokenizer,model_max_length=20)

{'input_ids': [23069,
  29,
  2705,
  11877,
  189,
  189,
  4122,
  15263,
  29,
  2705,
  11877,
  355,
  11137,
  34653,
  3515,
  4582,
  2,
  23069,
  29,
  2705],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [-100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  11877,
  355,
  11137,
  34653,
  3515,
  4582,
  2,
  -100,
  -100,
  -100]}

# data collator

    bloom是靠左填充!! 這不一樣!!

    填充字元在左側，文字在右側。

    參數padding表示填充方式，可以為布爾類型、字符串類型或者一個PaddingStrategy對象。
    當值為布爾類型時，True表示填充至最大序列長度，False表示不填充。
    當為字符串類型時，"longest"表示填充值最大序列長度，"max_length"表示填充值參數max_length設置的長度，"do_not_pad"表示不填充。  
    
    參數max_length表示填充序列的最大長度，當設置padding="max_length"時，該參數才會有用。  
    
    參數pad_to_multiple_of表示填充的序列的倍數。  

    參數label_pad_token_id表示填充標籤時的值，默認為-100。注意，默認數據中序列填充的值為0，這與標籤填充的值不一致。

    label_pad_token_id (int, *optional*, defaults to -100):

    data collator會進行批量化，每批會以最長文句的長度，不足長度的樣本會填充字元(pad_token_id)在左側。

In [20]:
#  label_pad_token_id (int, *optional*, defaults to -100):
data_collator = transformers.DataCollatorForSeq2Seq(tokenizer, return_tensors="pt", padding=True)

In [21]:
data_collator

DataCollatorForSeq2Seq(tokenizer=BloomTokenizerFast(name_or_path='Langboat/bloom-389m-zh', vocab_size=42437, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False), model=None, padding=True, max_length=None, pad_to_multiple_of=None, label_pad_token_id=-100, return_tensors='pt')

In [22]:
data_point1 =  {
        "id": "uniq_sample_id",
        "conversations": [
            {"from": "human", "value": "你好"},
            {"from": "assistant", "value": "你好，有什么可以帮助你的吗？"},
            {"from": "human", "value": "今天天气怎么样？"},
            {"from": "assistant", "value": "不好意思，我无法回答你的问题，因为我不知道你的位置信息，同时我目前还无法获取到最新的天气信息。"}
        ]
    }
data_point2 =  {
        "id": "uniq_sample_id",
        "conversations": [
            {"from": "human", "value": "如何學習程式?"},
            {"from": "assistant", "value": "你可以修讀資管系的程式設計課程"},
        ]
    }

sample_encoded1 =generate_and_tokenize_prompt(data_point=data_point1, tokenizer=tokenizer,model_max_length=80)
print(len(sample_encoded1['input_ids']))
sample_encoded2 =generate_and_tokenize_prompt(data_point=data_point2, tokenizer=tokenizer,model_max_length=80)
print(len(sample_encoded2['input_ids']))

55
22


In [23]:
encoded_batch = data_collator([sample_encoded1, sample_encoded2,])

You're using a BloomTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [24]:
encoded_batch

{'input_ids': tensor([[23069,    29,  2705, 11877,   189,   189,  4122, 15263,    29,  2705,
         11877,   355, 11137, 34653,  3515,  4582,     2, 23069,    29,  2705,
          6089, 17216, 28609,   189,   189,  4122, 15263,    29,  2705, 25199,
           355,   683,  5684, 10639,  3515,  1960,   355,  3047, 14202,  3515,
          6857,  3882,   355,  4232,   683,  4415,  1624,  5684, 12801,   851,
         24399, 17216,  3882,   420,     2],
        [    3,     3,     3,     3,     3,     3,     3,     3,     3,     3,
             3,     3,     3,     3,     3,     3,     3,     3,     3,     3,
             3,     3,     3,     3,     3,     3,     3,     3,     3,     3,
             3,     3,     3, 23069,    29,  2705,  4803, 10529, 12380,   774,
           189,  4122, 15263,    29,  2705, 10747,  2759,  6045,  3249,  1656,
         21610, 12380,  7522, 18004,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
     

In [25]:
encoded_batch['input_ids'].shape

torch.Size([2, 55])

In [26]:
tokenizer.decode([11877,   355, 11137, 34653,  3515,  4582,     2])

'你好，有什么可以帮助你的吗？</s>'

In [27]:
tokenizer.decode([25199,
           355,   683,  5684, 10639,  3515,  1960,   355,  3047, 14202,  3515,
          6857,  3882,   355,  4232,   683,  4415,  1624,  5684, 12801,   851,
         24399, 17216,  3882,   420,     2])

'不好意思，我无法回答你的问题，因为我不知道你的位置信息，同时我目前还无法获取到最新的天气信息。</s>'

In [28]:
tokenizer.decode([10747,  2759,  6045,  3249,  1656,
         21610, 12380,  7522, 18004,     2])

'你可以修讀資管系的程式設計課程</s>'

# Read, preprocess, and save dataset

In [28]:
train_file = "data-part2-dialog-train_3.5M_CN./train_3.5M_CN.json"
train_data = datasets.load_dataset("json", data_files=train_file)

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset json downloaded and prepared to C:/Users/clhuang/.cache/huggingface/datasets/json/default-254bfe8ba16a8f1e/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

In [29]:
train_data

DatasetDict({
    train: Dataset({
        features: ['conversations', 'id'],
        num_rows: 3606402
    })
})

In [30]:
# 大約10萬筆
train_test_data = train_data["train"].train_test_split(test_size=0.03, shuffle=True, seed=42)

# 大約5萬筆
# train_test_data = train_data["train"].train_test_split(test_size=0.015, shuffle=True, seed=42)

In [31]:
train_test_data

DatasetDict({
    train: Dataset({
        features: ['conversations', 'id'],
        num_rows: 3498209
    })
    test: Dataset({
        features: ['conversations', 'id'],
        num_rows: 108193
    })
})

In [32]:
# 測試資料只要大約2000筆即可
train_test_data = train_test_data["test"].train_test_split(test_size=0.02, shuffle=True, seed=42)

In [33]:
train_test_data

DatasetDict({
    train: Dataset({
        features: ['conversations', 'id'],
        num_rows: 106029
    })
    test: Dataset({
        features: ['conversations', 'id'],
        num_rows: 2164
    })
})

In [34]:
train_test_data['train'][0]

{'conversations': [{'from': 'human', 'value': '编写一篇有关皮肤护理的文章，包括护理技巧和必备护肤品。'},
  {'from': 'assistant',
   'value': '皮肤是我们身体最大的器官之一，对外界的刺激非常敏感，所以我们需要特别关注它的健康。以下是一些护肤技巧以及必备的护肤产品。\n1.每天保持清洁：每天早晚使用洁面乳或洗面奶洗脸，可以去除皮肤表层的污垢和油脂，让皮肤保持清爽干净。同时不要用力搓揉皮肤，以免导致皮肤受伤。\n2.注意去角质：去角质可以有效地去除皮肤表面的老废角质，让皮肤更容易吸收养分。但是，要遵循适当的方法和频率。通常建议每周去角质一次。\n3.使用加强护理的面霜：除了涂抹普通面霜之外，可以适时使用加强护理的面霜来滋润肌肤。例如有些面霜添加了多种营养成分、水杨酸等，有助于延缓皮肤老化。\n4.注意保湿：干燥是皮肤老化的罪魁祸首。需要做好皮肤的保湿工作来使皮肤健康。不同的皮肤类型需要不同的保湿方法和产品。\n5.防晒：阳光中的紫外线对皮肤损伤非常大，可以增加皮肤老化和皮肤癌的风险。因此，在日常护理中，不要忽视防晒。建议选择SPF值30或更高的防晒霜。\n上述皮肤护理技巧需要配合适用的护肤品，下面介绍几种必备的护肤品：\n1.洗面奶或洁面乳：洗面奶或洁面乳可以快速清洁皮肤，去除油脂和污垢。\n2.面霜：面霜具有保湿和滋润作用，能增强皮肤的抵抗力和防御能力。\n3.面膜：面膜能改善肌肤质地和颜色，滋润肌肤，提高皮肤的免疫力。\n4.眼霜：眼周区域非常脆弱，需要使用特殊的眼霜，防止因长期使用电子产品，长时间使用的影响影响细胞对于光原的信号的感知效应。\n总之，皮肤护理是我们日常护理中重要的一环。我们应该为自己的皮肤提供适量的保护和滋润。选择适合自己肤质的护肤品，合理的使用方法，才能真正护理好你的皮肤。'}],
 'id': '41973849'}

In [35]:
from functools import partial

In [36]:
model_max_length

512

In [37]:
val_data = train_test_data["test"].map(partial(generate_and_tokenize_prompt, model_max_length, tokenizer)).remove_columns(['id', 'conversations'])
#val_data = train_test_data["test"].map(generate_and_tokenize_prompt, model_max_length, tokenizer).remove_columns(['id', 'conversations'])
# val_data = (train_test_data["test"].shuffle().map(partial(generate_and_tokenize_prompt, model_max_length, tokenizer)))

Map:   0%|          | 0/2164 [00:00<?, ? examples/s]

In [38]:
val_data

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2164
})

In [39]:
train_test_data["test"][0]

{'conversations': [{'from': 'human', 'value': '给我讲个笑话吧。'},
  {'from': 'assistant', 'value': '知道为什么只有八个海盗能劈开柿子吗？因为有九个会被柿子砸中。'},
  {'from': 'human', 'value': '好玩！再给我想一个。'},
  {'from': 'assistant', 'value': '为什么水滴会掉下来？ 因为大气中的玄学力场增强了水滴表面张力，引起水滴落体现象。'},
  {'from': 'human', 'value': '打开音乐播放器，播放我最近最喜欢的歌。'},
  {'from': 'assistant',
   'value': '好的，请问您的音乐播放器是哪一个？我可以为您实现基于YouTube、Spotify或SoundCloud等各种音乐平台的播放操作。请告诉我您的需要。'},
  {'from': 'human', 'value': '给我推荐一个适合自学的网站。'},
  {'from': 'assistant',
   'value': 'Codecademy、Khan Academy、Youtube无疑是优秀的自学网站，Codecademy提供了丰富多样的编程教学资源，KhanAcademy的学科范围更为广泛，Youtube上的各种新闻、21世纪思想家、TED等等完全可以成为自学的好材料之一。'}],
 'id': '8665371'}

In [40]:
val_data[0]

{'input_ids': [26283,
  29,
  2813,
  9844,
  5674,
  907,
  36634,
  3127,
  672,
  189,
  4340,
  17245,
  29,
  2813,
  3015,
  7337,
  5178,
  34938,
  28648,
  975,
  29003,
  1390,
  26068,
  1020,
  4854,
  3174,
  620,
  45010,
  19823,
  26068,
  1020,
  21988,
  655,
  420,
  2,
  26283,
  29,
  2813,
  40373,
  2287,
  2188,
  2481,
  7282,
  1667,
  672,
  189,
  4340,
  17245,
  29,
  2813,
  7337,
  1669,
  13111,
  743,
  6079,
  9841,
  1518,
  18098,
  21958,
  3403,
  11331,
  1260,
  1276,
  2306,
  12144,
  658,
  1669,
  13111,
  11910,
  3685,
  1276,
  355,
  7236,
  1669,
  13111,
  3329,
  1458,
  9419,
  420,
  2,
  26283,
  29,
  2813,
  9473,
  7591,
  13539,
  2339,
  355,
  13539,
  685,
  8269,
  1277,
  24257,
  3833,
  672,
  189,
  4340,
  17245,
  29,
  2813,
  4775,
  355,
  33171,
  16845,
  7591,
  13539,
  2339,
  42770,
  1667,
  1518,
  16177,
  28100,
  5371,
  11091,
  24708,
  554,
  30153,
  3347,
  1326,
  18649,
  20543,
  1298,
  5924,
  

In [41]:
model_max_length=512 #
train_data = train_test_data["train"].map(partial(generate_and_tokenize_prompt, model_max_length, tokenizer)).remove_columns(['id', 'conversations'])


Map:   0%|          | 0/106029 [00:00<?, ? examples/s]

In [42]:

train_data.save_to_disk('./train_dataset/data_train')
val_data.save_to_disk('./train_dataset/data_val')

Saving the dataset (0/1 shards):   0%|          | 0/106029 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2164 [00:00<?, ? examples/s]

In [43]:
train_data = datasets.load_from_disk('train_dataset/data_train/')
val_data = datasets.load_from_disk('train_dataset/data_val/')

In [44]:
train_data

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 106029
})

In [45]:
val_data

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2164
})

In [46]:
# 平均長度有264
total=0
for item in train_data['input_ids'][:2000]:
    total += len(item)
total/2000

263.1035

# 參考資料

In [47]:
data_point={'conversation_id': 685986,
 'category': 'Role Playing',
 'conversation': [{'human': '仿照客服专员，处理一条投诉',
   'assistant': '您好，我是MOSS，很抱歉听到您的投诉。请问您可以告诉我具体的情况吗？我会尽力帮您解决问题。\n\n（等待顾客回复）\n\n我很抱歉您遇到这种情况。我可以理解您的不满和失望。请您放心，我会尽快处理并解决这个问题。\n\n首先，我会记录您的投诉并向相关部门反映。同时，我会尽快联系相关人员，了解情况，并寻找最佳解决方案。我会保持与您的联系，并及时向您反馈进展情况。\n\n如果您需要进一步的帮助或解释，我会尽全力协助您。非常感谢您对我们的支持和理解。'},
  {'human': '请问你们需要我的一些个人信息来帮我解决问题吗？',
   'assistant': '非常感谢您的关注。为了更好地帮助您解决问题，我们可能需要您提供一些个人信息，例如您的姓名、联系方式、订单号等。请放心，我们非常重视您的隐私和安全，您提供的信息将会被严格保密，并且只会用于解决您的问题。如果您对提供个人信息有任何疑虑或担忧，请随时告诉我，我们会尽力解答您的问题。'},
  {'human': '我希望能够了解你们的投诉处理流程，可以吗？',
   'assistant': '当然可以！我们的投诉处理流程主要包括以下几个步骤：\n\n1.收集投诉信息：我们会认真听取您的投诉内容，并收集您的个人信息和相关证据。\n\n2.反馈投诉处理进度：我们会及时向您反馈我们的处理进度，包括投诉是否已经被接受、处理人员的联系方式等。\n\n3.处理投诉：我们将对您的投诉内容进行调查和核实，并根据实际情况采取措施，解决您的问题。\n\n4.解决投诉：我们会尽力解决您的问题，并向您说明解决方案。\n\n5.跟进投诉：我们会在解决问题后进行跟进，确保您的问题得到彻底解决，同时收集您的反馈信息，以便我们持续改进服务质量。\n\n如果您对我们的投诉处理流程有任何疑问或建议，欢迎随时联系我们，我们将尽力提供满意的答复。'}
  ]}

In [48]:
# sft_sample_to_ids_v2(data_point["conversation"],tokenizer)